In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import ast
from sklearn.metrics import accuracy_score
import copy
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,average_precision_score)
from sklearn.neural_network import MLPClassifier

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from scipy.stats import spearmanr

# Load Data

In [5]:
newness = np.load("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\newness_2024.npy",allow_pickle=True)
surprise = np.load("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\surprise_2024.npy",allow_pickle=True)
value = np.load("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\value_2024.npy",allow_pickle=True)

In [8]:
rho, pval = spearmanr(newness,surprise)
print(f"The Correlation between newness and Surprise is {rho}")

The Correlation between newness and Surprise is 0.2985509641365827


In [9]:
rho, pval = spearmanr(newness,value)
print(f"The Correlation between newness and value is {rho}")

The Correlation between newness and value is 0.042086096035055594


In [10]:
rho, pval = spearmanr(value,surprise)
print(f"The Correlation between surprise and value is {rho}")

The Correlation between surprise and value is 0.030809796371711884


In [23]:
surprise = surprise.reshape(-1)

In [12]:
surprise.shape

(355551, 1)

In [13]:
BASELINE_YEAR=1600

In [14]:
gold_set_70 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_70.xlsx")

In [15]:
creative_70 = np.load("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\creativity_70.npy",allow_pickle=True)

In [16]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [17]:
merged_df = pd.merge(gold_set_70, artnet_2024[["artwork id","artist id","year born","workyear modifier","price","artist",'est lo usd','est hi usd', 'sale price usd']], on='artwork id', how='left')

In [18]:
merged_df.shape

(1084, 32)

In [19]:
merged_df.columns

Index(['artwork id', 'title', 'first', 'last', 'workyear from', 'nationality',
       'artistic_value_answer_claude', 'artistic_value_comment_claude',
       'creativity_answer_claude', 'creativity_comment_claude',
       'artistic_value_answer_gemini', 'artistic_value_comment_gemini',
       'creativity_answer_gemini', 'creativity_comment_gemini',
       'artistic_value_answer', 'artistic_value_comment', 'creativity_answer',
       'creativity_comment', 'creative_consist', 'artistic_consist',
       'overall_consist', 'embed_artistic_consist', 'embed_creative_consist',
       'embed_overall_consist', 'artist id', 'year born', 'workyear modifier',
       'price', 'artist', 'est lo usd', 'est hi usd', 'sale price usd'],
      dtype='object')

In [20]:
merged_df[merged_df["workyear from"]>BASELINE_YEAR].shape

(1084, 32)

In [21]:
row = merged_df.iloc[0]
row_index = row.index

In [22]:
merged_df.iloc[0]

artwork id                                                                    2849
title                                                                  HOMME ASSIS
first                                                                        Pablo
last                                                                       Picasso
workyear from                                                                 1969
nationality                                                                Spanish
artistic_value_answer_claude                                                  High
artistic_value_comment_claude    "Homme Assis" was painted during Picasso's mos...
creativity_answer_claude                                                       Yes
creativity_comment_claude        Picasso's objective to paint 'nature' contrast...
artistic_value_answer_gemini                                                  High
artistic_value_comment_gemini    Picasso's "Homme Assis" from 1969 is widely co...
crea

In [23]:
row_index

Index(['artwork id', 'title', 'first', 'last', 'workyear from', 'nationality',
       'artistic_value_answer_claude', 'artistic_value_comment_claude',
       'creativity_answer_claude', 'creativity_comment_claude',
       'artistic_value_answer_gemini', 'artistic_value_comment_gemini',
       'creativity_answer_gemini', 'creativity_comment_gemini',
       'artistic_value_answer', 'artistic_value_comment', 'creativity_answer',
       'creativity_comment', 'creative_consist', 'artistic_consist',
       'overall_consist', 'embed_artistic_consist', 'embed_creative_consist',
       'embed_overall_consist', 'artist id', 'year born', 'workyear modifier',
       'price', 'artist', 'est lo usd', 'est hi usd', 'sale price usd'],
      dtype='object')

In [24]:
newness_df = np.zeros(merged_df.shape[0], dtype=object)
surprise_df = np.zeros(merged_df.shape[0], dtype=object)
value_df = np.zeros(merged_df.shape[0], dtype=object)
for i in range(merged_df.shape[0]):
    row_index = merged_df.index[i]
    newness_df[i] =newness[row_index]
    surprise_df[i] =surprise[row_index]
    value_df[i] = value[row_index]

# Basic

Balanced + Normalized (max min)

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from scipy.stats import spearmanr

In [142]:
newness_min=np.min(newness_df)
newness_max = np.max(newness_df)
newness_normalized=(newness_df - newness_min)/(newness_max-newness_min)

In [143]:
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)

In [144]:
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)

In [145]:
eps = 1e-6

In [146]:
y = np.log(creative_70)
X = np.column_stack([
    np.log((newness_normalized+eps).astype(float)),
    np.log((surprise_normalized+eps).astype(float)),
    np.log((value_normalized+eps).astype(float))
])

In [147]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=23)

In [148]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [149]:
np.sum(y)/len(y)

np.float64(-0.059825041309513126)

In [150]:
np.sum(y_train)/len(y_train)

np.float64(-0.060014087187317204)

In [151]:
np.sum(y_test)/len(y_test)

np.float64(-0.05906972897745724)

In [1]:
mse = mean_squared_error(y_test, model.predict(X_test))
print(f"Test Linear Regression model mse: {mse:.4f}")
mae = mean_absolute_error(y_test, model.predict(X_test))
print("Test mae:", mae)
RMSE = root_mean_squared_error(y_test, model.predict(X_test))
print("Test RMSE:", RMSE)
rho, pval = spearmanr(y_test, model.predict(X_test))
print("Test spearman:", rho)

NameError: name 'mean_squared_error' is not defined

In [153]:
rate = 0.1
k = int(len(y_test) * rate)
true_top_idx = np.argsort(y_test)[-k:]
pred_top_idx = np.argsort(model.predict(X_test))[-k:]
hit_rate = len(set(true_top_idx) & set(pred_top_idx)) / k
print(f"Top {rate} Hit Rate:", hit_rate)

Top 0.1 Hit Rate: 0.0


# Splines 
Splines + Balanced + Normalized (max min)

In [116]:
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)
newness_min=np.min(newness_df)
newness_max = np.max(newness_df)
newness_normalized=(newness_df - newness_min)/(newness_max-newness_min)
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)

In [117]:
y = np.log(creative_70)
X = np.column_stack([
    np.log((newness_normalized+eps).astype(float)),
    np.log((surprise_normalized+eps).astype(float)),
    np.log((value_normalized+eps).astype(float))
])

In [126]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=23)

In [119]:
preprocess = ColumnTransformer([
    ("spline", SplineTransformer(
        degree=3,           # cubic
        n_knots=8,          # tune with CV (e.g., 5–10 common)
        extrapolation="linear",
        include_bias=False  # avoid intercept duplication
    ), [1,2]),
], remainder="drop")

In [122]:
model = make_pipeline(
    preprocess,
    LinearRegression()
)

In [127]:
model.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('spline',
                                                  SplineTransformer(extrapolation='linear',
                                                                    include_bias=False,
                                                                    n_knots=8),
                                                  [1, 2])])),
                ('linearregression', LinearRegression())])

In [139]:
mse = mean_squared_error(y_test, model.predict(X_test))
print(f"Test Linear Regression model mse: {mse:.4f}")
mae = mean_absolute_error(y_test, model.predict(X_test))
print("Test mae:", mae)
RMSE = root_mean_squared_error(y_test, model.predict(X_test))
print("Test RMSE:", RMSE)
rho, pval = spearmanr(y_test, model.predict(X_test))
print("Test spearman:", rho)

Test Linear Regression model mse: 0.0025
Test mae: 0.03219791140808735
Test RMSE: 0.04967567910791389
Test spearman: 0.09160923913809382
Top 0.1 Hit Rate: 0.047619047619047616


In [140]:
rate = 0.1
k = int(len(y_test) * rate)
true_top_idx = np.argsort(y_test)[-k:]
pred_top_idx = np.argsort(model.predict(X_test))[-k:]
hit_rate = len(set(true_top_idx) & set(pred_top_idx)) / k
print(f"Top {rate} Hit Rate:", hit_rate)

Top 0.1 Hit Rate: 0.047619047619047616


# Deep Neural Network

In [26]:
from sklearn.neural_network import MLPRegressor

In [33]:
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)
newness_min=np.min(newness_df)
newness_max = np.max(newness_df)
newness_normalized=(newness_df - newness_min)/(newness_max-newness_min)

In [34]:
surprise_normalized

array([np.float64(0.24170821867840708), np.float64(0.21776844805543694),
       np.float64(0.29997440953790294), ...,
       np.float64(0.32183254343082385), np.float64(0.2335193740051044),
       np.float64(0.36288648724003403)], shape=(1084,), dtype=object)

In [35]:
eps = 1e-6

In [36]:
y = np.log(creative_70)
X = np.column_stack([
    np.log((newness_normalized+eps).astype(float)),
    np.log((surprise_normalized+eps).astype(float)),
    np.log((value_normalized+eps).astype(float))
])

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=23)

In [70]:
model = MLPRegressor(solver='lbfgs', alpha=3e-5,
                    hidden_layer_sizes=(8, 8), random_state=1,max_iter=1000)
model.fit(X_train, y_train)

MLPRegressor(alpha=3e-05, hidden_layer_sizes=(8, 8), max_iter=1000,
             random_state=1, solver='lbfgs')

In [71]:
mse = mean_squared_error(y_test, model.predict(X_test))
print(f"Test Linear Regression model mse: {mse:.4f}")
mae = mean_absolute_error(y_test, model.predict(X_test))
print("Test mae:", mae)
RMSE = root_mean_squared_error(y_test, model.predict(X_test))
print("Test RMSE:", RMSE)
rho, pval = spearmanr(y_test, model.predict(X_test))
print("Test spearman:", rho)

Test Linear Regression model mse: 0.0026
Test mae: 0.033126829521831484
Test RMSE: 0.051454067590900834
Test spearman: 0.07771623989439913


In [72]:
rate = 0.1
k = int(len(y_test) * rate)
true_top_idx = np.argsort(y_test)[-k:]
pred_top_idx = np.argsort(model.predict(X_test))[-k:]
hit_rate = len(set(true_top_idx) & set(pred_top_idx)) / k
print(f"Top {rate} Hit Rate:", hit_rate)

Top 0.1 Hit Rate: 0.09523809523809523
